# Qwen3.8-27B + LangGraph Multi-Step API Orchestrator on Kaggle

End-to-end architecture running inside Kaggle:
1. **LLM Engine (`:8000`):** Prebuilt `llama.cpp` (`llama-server`) serving `Qwen3.8-27B` with `--jinja` tool-calling support.
2. **Downstream REST APIs (`:9000`):** Self-contained demo business API (`/customers`, `/orders`, `/tickets`) — swap `DOWNSTREAM_API_BASE` with your real APIs.
3. **LangGraph Gateway (`:8080`):** FastAPI + LangGraph (`StateGraph` + `ToolNode` + `MemorySaver`) sitting between clients and your APIs, exposed via **both** `/workflow` and an OpenAI-compatible `/v1/chat/completions`.
4. **Public Access:** Exposed via `cloudflared` tunnel.

### Kaggle Settings
- **Accelerator:** **GPU T4 x2** (recommended) or **GPU P100** (use `Qwen3.8-27B-UD-IQ3_S.gguf` if tight on 16 GB VRAM)
- **Internet:** **ON**

## 1. Configuration & Hardware Check

In [ ]:
import os, shutil, subprocess

HF_REPO = "unsloth/Qwen3.8-27B-GGUF"
HF_FILENAME = "Qwen3.8-27B-UD-IQ4_XS.gguf"
LOCAL_MODEL_PATH = ""  # Optional: "/kaggle/input/your-dataset/model.gguf"

CONTEXT_LEN = 8192
LLM_PORT = 8000
MOCK_API_PORT = 9000
GATEWAY_PORT = 8080
BIN_DIR = "/tmp/llama-bin"

candidates = [p for p in ["/tmp", "/kaggle/working"] if os.path.exists(p)]
STORAGE_ROOT = max(candidates, key=lambda p: shutil.disk_usage(p).free)
MODELS_DIR = os.path.join(STORAGE_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(BIN_DIR, exist_ok=True)

for p in candidates:
    print(f"Free disk at {p}: {shutil.disk_usage(p).free / (1024**3):.1f} GB")

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
HAS_GPU = gpu_check.returncode == 0
print("GPU:", gpu_check.stdout.strip() if HAS_GPU else "CPU-only")

## 2. Download Prebuilt `llama.cpp` (`llama-server`)

In [ ]:
import json, urllib.request, tarfile

req = urllib.request.Request(
    "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=20",
    headers={"User-Agent": "kaggle-notebook"}
)
with urllib.request.urlopen(req) as resp:
    releases = json.loads(resp.read().decode())

if HAS_GPU:
    def needed_for(tag):
        return [
            f"llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
            f"cudart-llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
        ]
else:
    def needed_for(tag):
        return [f"llama-{tag}-bin-ubuntu-x64.tar.gz"]

release = next(
    (
        r for r in releases
        if r.get("tag_name", "").startswith("b")
        and all(
            any(a["name"] == fn for a in r.get("assets", []))
            for fn in needed_for(r["tag_name"])
        )
    ),
    None,
)
if not release:
    raise RuntimeError("No recent llama.cpp release with matching prebuilt assets found.")

tag = release["tag_name"]
assets = {a["name"]: a["browser_download_url"] for a in release["assets"]}
print("Using llama.cpp release:", tag)

for fname in needed_for(tag):
    url = assets[fname]
    tgz_path = f"/tmp/{fname}"
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(url, tgz_path)
    with tarfile.open(tgz_path, "r:gz") as tf:
        for member in tf.getmembers():
            parts = member.name.split("/", 1)
            if len(parts) < 2 or not parts[1]:
                continue
            member.name = parts[1]
            tf.extract(member, BIN_DIR)
    os.remove(tgz_path)

LLAMA_SERVER = os.path.join(BIN_DIR, "llama-server")
os.chmod(LLAMA_SERVER, 0o755)
print("Ready:", LLAMA_SERVER)

## 3. Download Model & Install LangChain / LangGraph Packages

In [ ]:
!pip install -qU huggingface_hub langgraph langchain-openai langchain-core httpx fastapi uvicorn openai pydantic
from huggingface_hub import hf_hub_download

if LOCAL_MODEL_PATH and os.path.exists(LOCAL_MODEL_PATH):
    MODEL_PATH = LOCAL_MODEL_PATH
else:
    MODEL_PATH = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        local_dir=MODELS_DIR,
    )

print("Model path:", MODEL_PATH)

## 4. Start `llama-server` (LLM Backend on `:8000`)

In [ ]:
import time, urllib.request

subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(0.5)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{BIN_DIR}:{env.get('LD_LIBRARY_PATH', '')}"

cmd = [
    LLAMA_SERVER,
    "-m", MODEL_PATH,
    "--host", "127.0.0.1",
    "--port", str(LLM_PORT),
    "-c", str(CONTEXT_LEN),
    "-ngl", "99" if HAS_GPU else "0",
    "-fa", "on",
    "--jinja",
    "--alias", "qwen3.8-27b",
]

llm_log = "/tmp/llama_server.log"
llm_proc = subprocess.Popen(cmd, stdout=open(llm_log, "w"), stderr=subprocess.STDOUT, env=env)

for i in range(180):
    if llm_proc.poll() is not None:
        print(open(llm_log).read())
        raise RuntimeError(f"llama-server exited with code {llm_proc.returncode}")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{LLM_PORT}/health") as r:
            if r.status == 200:
                print(f"llama-server healthy in {i}s:", r.read().decode())
                break
    except Exception:
        time.sleep(1)
else:
    print(open(llm_log).read())
    raise TimeoutError("Timed out waiting for llama-server")

## 5. Start a Demo Downstream REST API (`:9000`)
In production, skip or replace this with your real API base URL (`DOWNSTREAM_API_BASE`).

In [ ]:
%%writefile /tmp/mock_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Demo Business API")

CUSTOMERS = {
    "C-101": {"id": "C-101", "name": "Asha Menon", "tier": "enterprise", "status": "active"},
    "C-102": {"id": "C-102", "name": "Ravi Kumar", "tier": "standard", "status": "suspended"},
}

ORDERS = {
    "ORD-500": {"order_id": "ORD-500", "customer_id": "C-101", "amount": 1299.00, "status": "delayed"},
    "ORD-501": {"order_id": "ORD-501", "customer_id": "C-102", "amount": 89.00, "status": "shipped"},
}

TICKETS = []

class TicketIn(BaseModel):
    customer_id: str
    priority: str
    summary: str

@app.get("/customers/{customer_id}")
def get_customer(customer_id: str):
    if customer_id not in CUSTOMERS:
        raise HTTPException(404, f"Customer {customer_id} not found")
    return CUSTOMERS[customer_id]

@app.get("/orders/{order_id}")
def get_order(order_id: str):
    if order_id not in ORDERS:
        raise HTTPException(404, f"Order {order_id} not found")
    return ORDERS[order_id]

@app.post("/tickets")
def create_ticket(t: TicketIn):
    if t.priority not in ("low", "medium", "high"):
        raise HTTPException(400, "priority must be low, medium, or high")
    ticket = {"ticket_id": f"TKT-{1000 + len(TICKETS)}", **t.model_dump()}
    TICKETS.append(ticket)
    return ticket

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn mock_api:app"], check=False)
time.sleep(0.5)

mock_proc = subprocess.Popen(
    ["uvicorn", "mock_api:app", "--host", "127.0.0.1", "--port", str(MOCK_API_PORT)],
    cwd="/tmp",
    stdout=open("/tmp/mock_api.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(2)
with urllib.request.urlopen(f"http://127.0.0.1:{MOCK_API_PORT}/customers/C-101") as r:
    print("Mock API ready:", r.read().decode())

## 6. Build the LangGraph Multi-Step Orchestrator + FastAPI Gateway (`:8080`)
Defines:
- Typed LangChain `@tool` wrappers around the REST endpoints.
- A `StateGraph` loop (`agent -> ToolNode -> agent`) with `handle_tool_errors=True` and `MemorySaver`.
- FastAPI endpoints: `/workflow` (returns trace + step count + final reply) and `/v1/chat/completions` (OpenAI-compatible facade).

In [ ]:
%%writefile /tmp/langgraph_gateway.py
import os
import time
import uuid
from typing import Annotated, List, Optional
from typing_extensions import TypedDict

import httpx
from fastapi import FastAPI
from pydantic import BaseModel

from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

LLM_BASE = os.environ.get("LLM_BASE", "http://127.0.0.1:8000/v1")
DOWNSTREAM_API_BASE = os.environ.get("DOWNSTREAM_API_BASE", "http://127.0.0.1:9000")

# --- 1. Wrap REST API endpoints as typed LangChain tools ---
@tool
def get_order(order_id: str) -> dict:
    """Fetch order details (including customer_id, amount, and status) by order_id (e.g. ORD-500)."""
    r = httpx.get(f"{DOWNSTREAM_API_BASE}/orders/{order_id}", timeout=10)
    r.raise_for_status()
    return r.json()

@tool
def get_customer(customer_id: str) -> dict:
    """Fetch customer profile, account tier (standard/enterprise), and status by customer_id (e.g. C-101)."""
    r = httpx.get(f"{DOWNSTREAM_API_BASE}/customers/{customer_id}", timeout=10)
    r.raise_for_status()
    return r.json()

@tool
def create_support_ticket(customer_id: str, priority: str, summary: str) -> dict:
    """Create a support ticket. priority must be exactly one of: 'low', 'medium', 'high'."""
    r = httpx.post(
        f"{DOWNSTREAM_API_BASE}/tickets",
        json={"customer_id": customer_id, "priority": priority, "summary": summary},
        timeout=10,
    )
    r.raise_for_status()
    return r.json()

tools = [get_order, get_customer, create_support_ticket]

# --- 2. Connect ChatOpenAI to local llama-server ---
llm = ChatOpenAI(
    base_url=LLM_BASE,
    api_key="not-needed",
    model="qwen3.8-27b",
    temperature=0.1,
)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = SystemMessage(
    content=(
        "You are an operations assistant with API tools. "
        "Always verify order and customer details via tools before creating tickets. "
        "For enterprise customers with delayed orders, set ticket priority to 'high'. "
        "Summarize all tool results concisely."
    )
)

# --- 3. Build LangGraph StateGraph ---
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def agent_node(state: AgentState):
    msgs = [SYSTEM_PROMPT] + state["messages"]
    response = llm_with_tools.invoke(msgs)
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph = builder.compile(checkpointer=MemorySaver())

# --- 4. Expose via FastAPI (custom /workflow + OpenAI /v1/chat/completions) ---
app = FastAPI(title="LangGraph + Qwen3.8-27B Orchestrator")

class WorkflowReq(BaseModel):
    query: str
    thread_id: str = "default"
    recursion_limit: int = 12

class ChatMsg(BaseModel):
    role: str
    content: str

class OpenAIChatReq(BaseModel):
    model: str = "qwen3.8-27b-agent"
    messages: List[ChatMsg]
    user: Optional[str] = "default"

@app.get("/health")
def health():
    return {"status": "ok", "llm_base": LLM_BASE, "api_base": DOWNSTREAM_API_BASE}

@app.post("/workflow")
async def run_workflow(req: WorkflowReq):
    out = await graph.ainvoke(
        {"messages": [("user", req.query)]},
        config={"configurable": {"thread_id": req.thread_id}, "recursion_limit": req.recursion_limit},
    )
    last_human_idx = max((i for i, m in enumerate(out["messages"]) if m.type == "human"), default=0)
    tool_trace = []
    for m in out["messages"][last_human_idx:]:
        if getattr(m, "tool_calls", None):
            tool_trace.append({"called": [tc["name"] for tc in m.tool_calls]})
        elif m.type == "tool":
            tool_trace.append({"tool": m.name, "output": m.content})
    return {
        "reply": out["messages"][-1].content,
        "steps": len(tool_trace),
        "trace": tool_trace,
    }

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": "qwen3.8-27b-agent", "object": "model"}]}

@app.post("/v1/chat/completions")
async def openai_chat(req: OpenAIChatReq):
    thread_id = req.user or "openai-default"
    last_user = next((m.content for m in reversed(req.messages) if m.role == "user"), "")
    out = await graph.ainvoke(
        {"messages": [("user", last_user)]},
        config={"configurable": {"thread_id": thread_id}, "recursion_limit": 12},
    )
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": out["messages"][-1].content},
                "finish_reason": "stop",
            }
        ],
    }

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn langgraph_gateway:app"], check=False)
time.sleep(0.5)

gw_env = os.environ.copy()
gw_env["LLM_BASE"] = f"http://127.0.0.1:{LLM_PORT}/v1"
gw_env["DOWNSTREAM_API_BASE"] = f"http://127.0.0.1:{MOCK_API_PORT}"

gw_proc = subprocess.Popen(
    ["uvicorn", "langgraph_gateway:app", "--host", "0.0.0.0", "--port", str(GATEWAY_PORT)],
    cwd="/tmp",
    stdout=open("/tmp/gateway.log", "w"),
    stderr=subprocess.STDOUT,
    env=gw_env,
)

for _ in range(20):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{GATEWAY_PORT}/health") as r:
            print("Gateway healthy:", r.read().decode())
            break
    except Exception:
        time.sleep(0.5)

## 7. Test Multi-Step Execution & Memory
This single query forces LangGraph + Qwen3.8-27B to execute **3 chained API calls**: `get_order(ORD-500)` -> `get_customer(C-101)` -> `create_support_ticket(high)`.

In [ ]:
import httpx, json

r = httpx.post(
    f"http://127.0.0.1:{GATEWAY_PORT}/workflow",
    json={
        "query": "Check order ORD-500. Look up the customer, and if the order is delayed, open a support ticket with the right priority.",
        "thread_id": "demo-thread-1",
    },
    timeout=180,
)
r.raise_for_status()
data = r.json()
print("Tool trace:\n", json.dumps(data["trace"], indent=2))
print("\nFinal reply:\n", data["reply"])

In [ ]:
# Test OpenAI SDK compatibility + thread memory (follow-up question on same thread)
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{GATEWAY_PORT}/v1", api_key="not-needed")
resp = client.chat.completions.create(
    model="qwen3.8-27b-agent",
    user="demo-thread-1",
    messages=[{"role": "user", "content": "What was the ticket ID and customer name from my previous request?"}],
)
print(resp.choices[0].message.content)

## 8. Expose the LangGraph Gateway Publicly via Cloudflare Tunnel

In [ ]:
%%bash
if [ ! -f /usr/local/bin/cloudflared ]; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi

cloudflared tunnel --url http://localhost:8080